$$
UCB(s_i)=\bar{V_i} + c\cdot \sqrt{\frac{\ln N}{n_i}},\quad c=2
$$
- node expansion
- rollout (random simulation)
    - 纯 MCTS：通常用随机走子或启发式轻量走子。
        - 完全依赖于大量的随机模拟，在复杂游戏中，随机模拟的指导性很差，导致搜索效率不高。
    - 带学习的 MCTS：用价值网络（事先训练好）直接给叶子估值，省去随机打穿。
- backpropagation

### UCB vs. UCT vs. PUCT

- UCB：多臂老虎机用的臂选择准则；不涉及树。
- UCT = UCB in Trees：把 UCB 的探索-利用权衡搬到 MCTS 的树节点上，完全数据驱动、无先验。
- PUCT = Prior + UCT：在 UCT 的探索项里引入策略先验 $P(s,a)$（通常来自规则/网络/启发式），能在大分支因子游戏上显著加速找到好招。

$$
\begin{align*}
\textbf{UCB:}\quad 
& \arg\max_{i}\;\Bigg[\, \overline{X}_i \;+\; c\,\sqrt{\frac{\ln n}{\,n_i\,}} \,\Bigg] \\[6pt]
\textbf{UCT:}\quad 
& \arg\max_{a}\;\Bigg[\, Q(s,a) \;+\; c\,\sqrt{\frac{\ln N(s)}{\,N(s,a)\,}} \,\Bigg] \\[6pt]
\textbf{PUCT:}\quad 
& \arg\max_{a}\;\Bigg[\, Q(s,a) \;+\; c_{\mathrm{puct}}\, P(s,a)\,
    \frac{\sqrt{\sum\limits_{b} N(s,b)}}{\,1+N(s,a)\,} \,\Bigg]
\end{align*}
$$

### pure inference vs. NN + MCTS

| 特性 | **纯MCTS** | **MCTS + 神经网络 (例如 AlphaGo)** |
| :--- | :--- | :--- |
| **工作模式** | 在推理时（下棋时）直接进行搜索和模拟 | 先训练神经网络，推理时用网络指导MCTS |
| **是否需要训练** | **否**，无需任何预训练 | **是**，需要大量自我对弈进行模型训练 |
| **核心驱动力** | 大量的随机模拟和统计 | 神经网络的“直觉”（策略网络）和“判断”（价值网络） |
| **性能** | 相对较弱，但通用性强 | 极为强大，是当前棋类AI的顶级水平 |
| **应用举例** | 一些早期的、相对简单的棋类AI程序 | AlphaGo、AlphaZero、Leela Chess Zero等 |

AI系统包含两个关键的神经网络，它们都需要预先进行大量的训练（通常是通过几百万盘的自我对弈）：
- 策略网络 (Policy Network)：它的作用是模仿“棋感”或“直觉”。输入一个棋局，它会输出一个概率分布，告诉你哪些走法看起来更有可能是高手的选择。
- 价值网络 (Value Network)：它的作用是进行“局势判断”。输入一个棋局，它会输出一个评估分数，预测当前局面下黑棋或白棋的最终胜率。

结合后的MCTS工作流程变为：
- 选择 (Selection)：与纯MCTS类似，但选择子节点时，除了考虑胜率和探索度，还会优先考虑策略网络推荐的走法。
    - PUCT
- 扩展 (Expansion)：当到达叶子节点时，不再随机扩展，而是根据策略网络的建议，优先扩展那些概率最高的走法。
- 模拟 (Simulation) 的替代：最关键的改变在这里。算法不再需要随机“下到死”，而是直接将当前叶子节点所代表的棋局输入价值网络，立即得到一个高质量的局势判断分数（例如，白方胜率65%）。这个分数将代替随机模拟的结果。
- 反向传播 (Backpropagation)：将价值网络给出的评估分数，反向传播更新路径上所有节点的统计数据。